# Sparse autoencoders: recover features from superposition
**LLMs From Basics · IN.3 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/in-3-sparse-autoencoder.ipynb)

Real LLM activations have no answer key: nobody knows the "true" features. So here you'll build a world where you *do* know them, in the spirit of *Toy Models of Superposition* (Elhage et al., 2022), and check whether a sparse autoencoder finds them:

1. **generate data**: 64 known feature directions squeezed into 32 dimensions, each feature rarely active
2. write the **SAE forward pass**: $f = \mathrm{ReLU}(W_{\text{enc}}(x - b_{\text{dec}}) + b_{\text{enc}})$, $\hat{x} = W_{\text{dec}} f + b_{\text{dec}}$
3. write the **SAE loss**: $\lVert x - \hat{x}\rVert^2 + \lambda \sum_i |f_i|$
4. **measure recovery**: match learned decoder directions to the true features by cosine similarity
5. sweep $\lambda$ to see the **sparsity–reconstruction trade-off**

Everything runs on CPU in well under a minute.

**How to use it:** run cells top to bottom (Shift+Enter). Each `# TODO` cell is followed by a ✅ check cell. Stuck for more than five minutes? The **Solutions** are at the bottom.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_num_threads(2)
d, n_true, m = 32, 64, 128        # activation size, true features, SAE latents
p_active = 0.05                   # each true feature is on 5% of the time

g = torch.Generator().manual_seed(0)
D_true = torch.randn(n_true, d, generator=g)
D_true = D_true / D_true.norm(dim=1, keepdim=True)   # 64 unit-length feature directions in 32-D

cos = D_true @ D_true.T - torch.eye(n_true)
print("true features:", tuple(D_true.shape), " largest |cos| between two of them:", round(cos.abs().max().item(), 2))

## 1. Generate data in superposition
Each data point is a sparse combination of the true directions: $x = \sum_i c_i\, d_i$.

* $c$ has shape `(N, 64)`: each entry is **on** with probability `p_active`, and when on its value is uniform on $[0.5, 1.5]$; otherwise it is 0.
* $x = c\, D_{\text{true}}$ has shape `(N, 32)`.

Use `torch.rand(..., generator=gen)` for both the on/off draw and the strength. Return `(x, c)`.

In [ ]:
def make_data(N, gen):
    # TODO: sample sparse coefficients c (N, n_true) and return (x, c) with x = c @ D_true.
    raise NotImplementedError("Fill in make_data")

In [ ]:
X, C = make_data(60000, torch.Generator().manual_seed(1))
X_test, C_test = make_data(10000, torch.Generator().manual_seed(2))
assert X.shape == (60000, d) and C.shape == (60000, n_true)
active = (C > 0).float().sum(1).mean().item()
assert 2.8 < active < 3.6, active                       # 64 x 0.05 = 3.2 on average
on = C[C > 0]
assert on.min() >= 0.5 and on.max() <= 1.5
assert torch.allclose(X, C @ D_true, atol=1e-5)
print(f"✅ data OK: {active:.2f} true features active per input, in only {d} dimensions")

## 2. The SAE forward pass
The decoder's columns are the learned feature directions. We store `W_dec` as a `(m, d)` matrix, one **row** per latent, and keep each row at unit length (`normalize_` is called after every optimizer step).

$$f = \mathrm{ReLU}\big((x - b_{\text{dec}})\, W_{\text{enc}} + b_{\text{enc}}\big), \qquad \hat{x} = f\, W_{\text{dec}} + b_{\text{dec}}$$

(Written for row vectors, as PyTorch batches are.) Implement `forward`, returning `(x_hat, f)`.

In [ ]:
class SAE(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.W_enc = nn.Parameter(torch.randn(d, m) * 0.1)
        self.b_enc = nn.Parameter(torch.zeros(m))
        self.W_dec = nn.Parameter(self.W_enc.data.T.clone())   # start tied to the encoder
        self.b_dec = nn.Parameter(torch.zeros(d))
        self.normalize_()

    @torch.no_grad()
    def normalize_(self):
        self.W_dec /= self.W_dec.norm(dim=1, keepdim=True)

    def forward(self, x):
        # TODO: compute latents f and reconstruction x_hat; return (x_hat, f)
        raise NotImplementedError("Fill in SAE.forward")

In [ ]:
torch.manual_seed(0)
sae = SAE(d, m)
xh, f = sae(X[:5])
assert xh.shape == (5, d) and f.shape == (5, m)
assert (f >= 0).all(), "ReLU latents can't be negative"
with torch.no_grad():
    sae.b_dec.fill_(0.3)
    xh2, f2 = sae(torch.full((1, d), 0.3))
assert torch.allclose(f2, torch.relu(sae.b_enc).unsqueeze(0)), "subtract b_dec before encoding"
assert torch.allclose(xh2, f2 @ sae.W_dec + 0.3), "add b_dec back after decoding"
assert torch.allclose(sae.W_dec.norm(dim=1), torch.ones(m))
print("✅ forward pass OK")

## 3. The loss
$$\mathcal{L} = \underbrace{\lVert x - \hat{x}\rVert_2^2}_{\text{reconstruction}} + \lambda \underbrace{\textstyle\sum_i |f_i|}_{\text{L1 sparsity}}$$

Sum over the dimensions of each example, then **average over the batch**. Return `(total, recon, l1)` where `recon` and `l1` are the batch means of the two terms (before multiplying by $\lambda$).

In [ ]:
def sae_loss(x, x_hat, f, lam):
    # TODO: return (total, recon, l1)
    raise NotImplementedError("Fill in sae_loss")

In [ ]:
# the worked example from the lesson: x = [1.0, -0.5, 0.2], x_hat = [0.9, -0.4, 0.2], f = [0, 1.2, 0, 0.5, 0, 0], λ = 0.01
x = torch.tensor([[1.0, -0.5, 0.2]]); x_hat = torch.tensor([[0.9, -0.4, 0.2]])
f = torch.tensor([[0, 1.2, 0, 0.5, 0, 0]])
total, recon, l1 = sae_loss(x, x_hat, f, 0.01)
assert abs(recon.item() - 0.02) < 1e-6 and abs(l1.item() - 1.7) < 1e-6
assert abs(total.item() - 0.037) < 1e-6
# batch averaging
total2, _, _ = sae_loss(torch.cat([x, x]), torch.cat([x_hat, x]), torch.cat([f, 0 * f]), 0.01)
assert abs(total2.item() - 0.0185) < 1e-6
print("✅ loss OK: 0.02 + 0.01 × 1.7 = 0.037")

## Train an SAE
This training loop is given. It uses your `SAE`, `sae_loss` and data. Adam, batches of 512, 1,500 steps, and the decoder rows are re-normalised after every step (otherwise the SAE could cheat the L1 penalty by shrinking $f$ and growing $W_{\text{dec}}$).

In [ ]:
def train_sae(lam, steps=1500, seed=0, m=m):
    torch.manual_seed(seed)
    sae = SAE(d, m)
    opt = torch.optim.Adam(sae.parameters(), lr=5e-3)
    for t in range(steps):
        x = X[torch.randint(0, X.shape[0], (512,))]
        x_hat, f = sae(x)
        loss, _, _ = sae_loss(x, x_hat, f, lam)
        opt.zero_grad(); loss.backward(); opt.step()
        sae.normalize_()
    return sae

t0 = time.time()
sae = train_sae(lam=0.2)
with torch.no_grad():
    xh, f = sae(X_test)
print(f"trained in {time.time() - t0:.1f}s; active latents per input: {(f > 0).float().sum(1).mean():.1f} (true: 3.2)")

## 4. Did it find the true features?
For each **true** feature $d_i$, find the learned decoder row with the highest cosine similarity. If the SAE worked, every true feature should have a near-perfect match.

Implement `best_match(W_dec, D_true)`: return a tensor of length 64 with each true feature's best cosine over the SAE's rows. (Normalise rows first, then one matrix product does all the cosines.)

In [ ]:
def best_match(W_dec, D_true):
    # TODO: (n_true,) tensor: for each true feature, the max cosine similarity over rows of W_dec
    raise NotImplementedError("Fill in best_match")

In [ ]:
assert torch.allclose(best_match(D_true, D_true), torch.ones(n_true), atol=1e-5)
assert torch.allclose(best_match(3 * D_true, D_true), torch.ones(n_true), atol=1e-5), "normalise the rows"
with torch.no_grad():
    bm = best_match(sae.W_dec, D_true)
recovered = int((bm > 0.9).sum())
print(f"mean best cosine {bm.mean():.3f}; recovered {recovered}/64 true features (cosine > 0.9)")
assert recovered >= 55, "with λ = 0.2 the SAE should recover nearly all features"

# compare: the raw neuron basis is a terrible dictionary
print(f"for comparison, the 32 neuron axes match the true features with mean best cosine {best_match(torch.eye(d), D_true).mean():.2f}")
print("✅ feature recovery OK")

## 5. The sparsity–reconstruction trade-off
Implement `evaluate(sae, X)` returning two numbers:

* `l0`: the average number of **non-zero** latents per input
* `nmse`: reconstruction error divided by the data's variance, $\dfrac{\text{mean}_n \lVert x_n - \hat{x}_n\rVert^2}{\text{mean}_n \lVert x_n - \bar{x}\rVert^2}$ (0 = perfect, 1 = no better than predicting the mean)

Then the cell after it trains five SAEs and plots the trade-off.

In [ ]:
@torch.no_grad()
def evaluate(sae, X):
    # TODO: return (l0, nmse) as Python floats
    raise NotImplementedError("Fill in evaluate")

In [ ]:
t0 = time.time()
results = []
for lam in [0.0, 0.03, 0.1, 0.3, 1.0]:
    s = train_sae(lam)
    l0, nmse = evaluate(s, X_test)
    rec = int((best_match(s.W_dec.detach(), D_true) > 0.9).sum())
    results.append((lam, l0, nmse, rec))
    print(f"λ = {lam:<4}  L0 = {l0:5.1f}   error = {100 * nmse:5.1f}% of variance   recovered {rec}/64")
print(f"({time.time() - t0:.0f}s)")

l0s = [r[1] for r in results]; errs = [r[2] for r in results]
assert all(a > b for a, b in zip(l0s, l0s[1:])), "more λ should mean fewer active latents"
assert errs[-1] > errs[0], "more λ should mean worse reconstruction"
assert results[0][3] < 20 and results[3][3] >= 55

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(l0s, [100 * e for e in errs], "o-")
for lam, l0, e, rec in results:
    ax.annotate(f"λ={lam}\n{rec}/64", (l0, 100 * e), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.axvline(3.2, ls="--", c="tab:orange", label="true features per input")
ax.set_xscale("log"); ax.set_xlabel("active latents per input (L0)"); ax.set_ylabel("reconstruction error (% of variance)")
ax.legend(); plt.tight_layout(); plt.show()
print("✅ trade-off OK")

**What you should see.** With $\lambda = 0$ the SAE reconstructs almost perfectly but smears each input over ~90 latents, and almost none of them lines up with a true feature: perfect reconstruction is not the goal. Moderate $\lambda$ recovers all 64 directions. Very large $\lambda$ starts to cost reconstruction (the L1 penalty also *shrinks* the active latents, one reason OpenAI switched to TopK SAEs). On a real LLM you see the same curve, but there's no answer key to tell you where the "right" point is.

## Experiment
Try a few of these, re-running the cells above:

* **Dictionary size.** Train with `m=64` (exactly the number of true features) and `m=512`. Does recovery change? Do you see duplicate latents for the same feature (a toy version of *feature splitting*) or latents that never fire (*dead latents*)? Count dead latents with `((f > 0).sum(0) == 0).sum()`.
* **Density.** Set `p_active = 0.2` and regenerate the data. With ~13 features active at once in 32 dimensions, superposition gets much harder: what happens to recovery?
* **TopK.** Replace the ReLU with a TopK activation: keep the `k = 4` largest pre-activations per row and zero the rest (`torch.topk`), and train with `lam = 0`. How does it compare with L1 at a similar L0?

---
## Solutions
Try the exercises first. The solutions below are one way to do it.

In [ ]:
# Solution: make_data
def make_data(N, gen):
    on = torch.rand(N, n_true, generator=gen) < p_active
    strength = 0.5 + torch.rand(N, n_true, generator=gen)       # uniform on [0.5, 1.5]
    c = on * strength
    return c @ D_true, c

In [ ]:
# Solution: SAE.forward (paste into the class)
class SAE(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.W_enc = nn.Parameter(torch.randn(d, m) * 0.1)
        self.b_enc = nn.Parameter(torch.zeros(m))
        self.W_dec = nn.Parameter(self.W_enc.data.T.clone())
        self.b_dec = nn.Parameter(torch.zeros(d))
        self.normalize_()

    @torch.no_grad()
    def normalize_(self):
        self.W_dec /= self.W_dec.norm(dim=1, keepdim=True)

    def forward(self, x):
        f = torch.relu((x - self.b_dec) @ self.W_enc + self.b_enc)
        x_hat = f @ self.W_dec + self.b_dec
        return x_hat, f

In [ ]:
# Solution: sae_loss
def sae_loss(x, x_hat, f, lam):
    recon = ((x - x_hat) ** 2).sum(dim=1).mean()
    l1 = f.abs().sum(dim=1).mean()
    return recon + lam * l1, recon, l1

In [ ]:
# Solution: best_match
def best_match(W_dec, D_true):
    W = W_dec / W_dec.norm(dim=1, keepdim=True)
    T = D_true / D_true.norm(dim=1, keepdim=True)
    return (T @ W.T).max(dim=1).values

In [ ]:
# Solution: evaluate
@torch.no_grad()
def evaluate(sae, X):
    x_hat, f = sae(X)
    l0 = (f > 0).float().sum(dim=1).mean().item()
    err = ((X - x_hat) ** 2).sum(dim=1).mean()
    var = ((X - X.mean(dim=0)) ** 2).sum(dim=1).mean()
    return l0, (err / var).item()